# -----------------------------------------------------------
#       Make a copy of this notebook in your own Drive if you want to keep your results or changes!
# ----------------------------------------------------------

#


# Protein Backbone Generation with RFdiffusion

## Overview

This notebook is a hands-on demo of **RFdiffusion** (https://www.nature.com/articles/s41586-023-06415-8), a diffusion model that generates novel protein **backbone structures** — either completely unconditionally ("from noise"), or conditioned at inference time on a fixed structural motif (e.g. an enzyme's active site) that must be reproduced somewhere inside a new, otherwise entirely novel, protein scaffold.

**This is not fine-tuning.** RFdiffusion is used exactly as released — all of its structural knowledge comes from its own pretraining. We only use its built-in **inference-time conditioning**: a short `contigs` string that tells the model which residues (if any) to hold fixed in 3D space, and which residues to design completely from scratch around them. No training happens in this notebook.

Adapted from Sergey Ovchinnikov's [ColabDesign RFdiffusion notebook](https://github.com/sokrypton/ColabDesign/blob/main/rf/examples/diffusion.ipynb), which already handles weight loading, the SE(3)-Transformer build, and the 3D trajectory animation reused below.

---

## ⚠️ This notebook needs a GPU (unlike the CLEAN and BOES notebooks)

RFdiffusion's denoising loop is a real neural network forward pass at every one of ~50 diffusion steps — on CPU this is dramatically slower. **Before running anything below:**

Runtime → Change runtime type → Hardware accelerator → **GPU** (a free T4 is enough) → Save

With a GPU, each generation below should take roughly 1-3 minutes. On CPU, expect the same run to take much longer (potentially tens of minutes), and installation itself also takes a few minutes regardless of hardware.

---

## What is RFdiffusion?

RFdiffusion adapts **RoseTTAFold** (a protein structure *prediction* network) into a structure *generation* network, by training it to reverse a noising process applied to real protein backbones — similar in spirit to image diffusion models (e.g. Stable Diffusion), but operating on 3D coordinates and orientations instead of pixels. Starting from random noise, the model iteratively denoises a set of residue positions and orientations until they form a plausible protein backbone.

Because it is fundamentally a *conditional* generator, RFdiffusion can be steered at inference time in several ways without any retraining:
- **Unconditional generation** — no constraints, just "generate a plausible protein of this length."
- **Motif scaffolding** — fix a known functional motif (e.g. a binding site or catalytic residues) in place, and design a novel scaffold around it.
- **Binder design, symmetric oligomers, fold conditioning**, and more (not covered in this short demo — see the original ColabDesign notebook for those).

## Why is generating a protein backbone hard?

- **The design space is astronomically large.** Even a modest 100-residue backbone has a continuous 3D conformational space; only a tiny fraction of it folds into a stable, well-packed structure at all.
- **Local plausibility isn't enough.** A model has to get side-chain packing, secondary structure, and long-range tertiary contacts simultaneously right — a backbone that looks fine locally can still be globally unfoldable.
- **Function is even harder than fold.** A folded scaffold is necessary but not sufficient — for an *enzyme*, the catalytic residues also need to end up in the right relative 3D geometry, which is exactly what motif scaffolding targets directly.

## Why hen egg-white lysozyme as the motif-scaffolding example?

We deliberately picked an enzyme people are likely to have heard of, and one that is almost certainly extremely well-represented in RFdiffusion's (and the underlying RoseTTAFold's) training data: hen egg-white lysozyme was the **first enzyme structure ever solved by X-ray crystallography**, and has been used as a crystallography and biophysics test protein for decades — the PDB contains dozens of essentially this same structure. Its catalytic residues, **Glu35** and **Asp52**, are independently verified below against a real deposited PDB entry rather than taken on faith.

---

## Checkpoints used in this notebook

RFdiffusion ships several checkpoints fine-tuned for different tasks. This notebook uses two of them:

- **`Base_ckpt.pt`** — the general-purpose model, used below for **unconditional generation**.
- **`ActiveSite_ckpt.pt`** — fine-tuned specifically for scaffolding *very small* functional motifs such as enzyme active sites. Per RFdiffusion's own documentation: *"for scaffolding minimalist sites such as enzyme active sites, we fine-tuned RFdiffusion on examples similar to these tasks, allowing it to hold smaller motifs better in place."* Used below for the **lysozyme active-site scaffolding** part.

Both checkpoints are mirrored at [soldatmat/CZAI_Summer_School-RFdiffusion_weights](https://huggingface.co/datasets/soldatmat/CZAI_Summer_School-RFdiffusion_weights) on Hugging Face — a fast, reliable copy of the official checkpoints. The official host, `files.ipd.uw.edu` (a University of Washington file server), can be slow or briefly unreachable, which is exactly the kind of risk we want to avoid when many students download it at the same time during a live lecture; if the mirror is ever unavailable, the setup cell below falls back to the official host automatically.


> **Note on the checkpoint weights used below.** This notebook downloads `Base_ckpt.pt`
> and `ActiveSite_ckpt.pt` from our own Hugging Face mirror,
> [`soldatmat/CZAI_Summer_School-RFdiffusion_weights`](https://huggingface.co/datasets/soldatmat/CZAI_Summer_School-RFdiffusion_weights),
> rather than the official `files.ipd.uw.edu` host, which has been unreliable/unreachable.
> **These are not byte-identical to the official release** -- they come from a third-party
> mirror and their MD5 checksums differ from the official ones (both are documented, and
> checked, in the download cell below). We're using them anyway because RFdiffusion's BSD
> license explicitly permits redistributing the weights, and because we independently
> verified these specific files are functionally correct: they load `strict=True` into the
> real model architecture with no NaNs, and a CPU test run using them to hold the lysozyme
> Glu35/Asp52 active site in place reproduced that geometry to within a fraction of an
> angstrom (see the Discussion at the end of this notebook). Full provenance and checksums
> are in the mirror's README.

In [ ]:
#@title Setup: install RFdiffusion, ColabDesign, and download weights (~2-4 min)
%%time
import os, sys, hashlib

if not os.path.isdir("RFdiffusion"):
    print("installing RFdiffusion...")
    os.system("git clone https://github.com/sokrypton/RFdiffusion.git")
    os.system("pip install -q jedi omegaconf hydra-core icecream pyrsistent pynvml decorator")
    os.system("pip install -q git+https://github.com/NVIDIA/dllogger#egg=dllogger")
    os.system("pip install -q --no-dependencies dgl -f https://data.dgl.ai/wheels/torch-2.4/cu124/repo.html")
    os.system("pip install -q --no-dependencies e3nn==0.5.5 opt_einsum_fx")
    os.system("cd RFdiffusion/env/SE3Transformer; pip install -q .")

if not os.path.isdir("colabdesign"):
    print("installing ColabDesign...")
    os.system("pip -q install git+https://github.com/sokrypton/ColabDesign.git")
    os.system("ln -s /usr/local/lib/python3.*/dist-packages/colabdesign colabdesign")

if not os.path.isdir("RFdiffusion/models"):
    print("downloading RFdiffusion weights...")
    os.makedirs("RFdiffusion/models", exist_ok=True)

    # Expected MD5, keyed by WHERE the file came from -- not a single hardcoded
    # hash. Our HF mirror currently hosts an alternate-source copy of these
    # checkpoints whose bytes differ from the official files.ipd.uw.edu release
    # (see the markdown note above and the dataset's README for why), so the two
    # sources have two different correct checksums. If the official host ever
    # comes back and gets used as the fallback, its download must be checked
    # against the OFFICIAL hash, not the mirror's -- otherwise a perfectly good
    # official download would look like a checksum failure.
    OFFICIAL_MD5 = {
        "Base_ckpt.pt": "6f5902ac237024bdd0c176cb93063dc4",
        "ActiveSite_ckpt.pt": "5532d2e1f3a4738decd58b19d633b3c3",
    }
    HF_MIRROR_MD5 = {
        "Base_ckpt.pt": "4aa4a27ba280d23541e01860c106c7cc",
        "ActiveSite_ckpt.pt": "0d9f82af03c73011c6fec060bac5b731",
    }
    EXPECTED_MD5_BY_SOURCE = {
        "hf_mirror": HF_MIRROR_MD5,
        "official": OFFICIAL_MD5,
    }
    OFFICIAL_URL = {
        name: f"http://files.ipd.uw.edu/pub/RFdiffusion/{md5}/{name}"
        for name, md5 in OFFICIAL_MD5.items()
    }
    HF_WEIGHTS_REPO = "soldatmat/CZAI_Summer_School-RFdiffusion_weights"

    def md5sum(path):
        h = hashlib.md5()
        with open(path, "rb") as f:
            for chunk in iter(lambda: f.read(1 << 20), b""):
                h.update(chunk)
        return h.hexdigest()

    from huggingface_hub import hf_hub_download

    for ckpt in OFFICIAL_MD5:
        dest = f"RFdiffusion/models/{ckpt}"
        try:
            cached = hf_hub_download(repo_id=HF_WEIGHTS_REPO, repo_type="dataset", filename=ckpt)
            os.system(f"cp {cached} {dest}")
            source = "hf_mirror"
            print(f"  {ckpt}: downloaded from the HF mirror")
        except Exception as e:
            print(f"  {ckpt}: HF mirror unavailable ({e}); falling back to files.ipd.uw.edu ...")
            os.system(f"wget -q -O {dest} {OFFICIAL_URL[ckpt]}")
            source = "official"

        expected_md5 = EXPECTED_MD5_BY_SOURCE[source][ckpt]
        got_md5 = md5sum(dest)
        ok = "OK" if got_md5 == expected_md5 else "MISMATCH -- re-download recommended"
        print(f"  {ckpt}: source={source} md5={got_md5} (expected {expected_md5} for this source) [{ok}]")

    # Cached IGSO(3) noise schedules speed up the first diffusion run; this is
    # a best-effort optimization only -- if it's unavailable, RFdiffusion just
    # computes the schedules itself (a bit slower) the first time it's used.
    os.system(
        "wget -q https://files.ipd.uw.edu/krypton/schedules.zip "
        "&& unzip -q -o schedules.zip -d RFdiffusion && rm -f schedules.zip"
    )

if "RFdiffusion" not in sys.path:
    os.environ["DGLBACKEND"] = "pytorch"
    sys.path.append("RFdiffusion")

import torch
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, HTML
import ipywidgets as widgets
import py3Dmol
from string import ascii_uppercase, ascii_lowercase

from inference.utils import parse_pdb
from colabdesign.rf.utils import get_ca, get_Ls, fix_contigs, fix_pdb, make_animation
from colabdesign.shared.protein import pdb_to_string
from colabdesign.shared.plot import plot_pseudo_3D, pymol_color_list

alphabet_list = list(ascii_uppercase + ascii_lowercase)

print("\nPyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU detected. Runtime -> Change runtime type -> GPU, then re-run this cell.")

In [ ]:
#@title Helper functions for running RFdiffusion (adapted from ColabDesign) { display-mode: "form" }
import time, random, string, signal

def get_pdb(pdb_code):
    """Fetch a 4-letter PDB code's biological assembly (used for the lysozyme motif below)."""
    if os.path.isfile(pdb_code):
        return pdb_code
    if not os.path.isfile(f"{pdb_code}.pdb1"):
        os.system(f"wget -qnc https://files.rcsb.org/download/{pdb_code}.pdb1.gz")
        os.system(f"gunzip -f {pdb_code}.pdb1.gz")
    return f"{pdb_code}.pdb1"


def run(command, steps, num_designs=1, visual="none"):
    """Launch run_inference.py and show a live progress bar (+ optional live preview)
    by watching the per-step PDB dumps RFdiffusion writes to /dev/shm."""

    def run_command_and_get_pid(command):
        pid_file = "/dev/shm/pid"
        os.system(f"nohup {command} & echo $! > {pid_file}")
        with open(pid_file, "r") as f:
            pid = int(f.read().strip())
        os.remove(pid_file)
        return pid

    def is_process_running(pid):
        try:
            os.kill(pid, 0)
        except OSError:
            return False
        return True

    run_output = widgets.Output()
    progress = widgets.FloatProgress(min=0, max=1, description="running", bar_style="info")
    display(widgets.VBox([progress, run_output]))

    for n in range(steps):
        if os.path.isfile(f"/dev/shm/{n}.pdb"):
            os.remove(f"/dev/shm/{n}.pdb")

    pid = run_command_and_get_pid(command)
    try:
        fail = False
        for _ in range(num_designs):
            for n in range(steps):
                wait = True
                while wait and not fail:
                    time.sleep(0.1)
                    if os.path.isfile(f"/dev/shm/{n}.pdb"):
                        pdb_str = open(f"/dev/shm/{n}.pdb").read()
                        if pdb_str[-3:] == "TER":
                            wait = False
                        elif not is_process_running(pid):
                            fail = True
                    elif not is_process_running(pid):
                        fail = True
                if fail:
                    progress.bar_style = "danger"
                    progress.description = "failed"
                    break
                progress.value = (n + 1) / steps
                if visual != "none":
                    with run_output:
                        run_output.clear_output(wait=True)
                        if visual == "image":
                            xyz, bfact = get_ca(f"/dev/shm/{n}.pdb", get_bfact=True)
                            fig = plt.figure()
                            fig.set_dpi(100); fig.set_figwidth(6); fig.set_figheight(6)
                            ax1 = fig.add_subplot(111); ax1.set_xticks([]); ax1.set_yticks([])
                            plot_pseudo_3D(xyz, c=bfact, cmin=0.5, cmax=0.9, ax=ax1)
                            plt.show()
                        elif visual == "interactive":
                            view = py3Dmol.view(js="https://3dmol.org/build/3Dmol.js")
                            view.addModel(pdb_str, "pdb")
                            view.setStyle({"cartoon": {"colorscheme": {"prop": "b", "gradient": "roygb", "min": 0.5, "max": 0.9}}})
                            view.zoomTo()
                            view.show()
                if os.path.exists(f"/dev/shm/{n}.pdb"):
                    os.remove(f"/dev/shm/{n}.pdb")
            if fail:
                progress.bar_style = "danger"
                progress.description = "failed"
                break
        while is_process_running(pid):
            time.sleep(0.1)
    except KeyboardInterrupt:
        os.kill(pid, signal.SIGTERM)
        progress.bar_style = "danger"
        progress.description = "stopped"


def run_diffusion(name, contigs, pdb=None, iterations=50, num_designs=1,
                   ckpt_override=None, visual="none"):
    """Run RFdiffusion. `contigs` follows ColabDesign's mini-language:
         '100'                              -> free 100-residue monomer (unconditional)
         '30-40/A33-37/8-14/A50-54/30-40'   -> two fixed motif windows taken from `pdb`,
                                                with new residues freely diffused around
                                                and between them (motif scaffolding)
    Returns (path, resolved_contigs) for use by show_trajectory().
    """
    path = name
    while os.path.exists(f"outputs/{path}_0.pdb"):
        path = name + "_" + "".join(random.choices(string.ascii_lowercase + string.digits, k=5))
    full_path = f"outputs/{path}"
    os.makedirs(full_path, exist_ok=True)

    opts = [f"inference.output_prefix={full_path}", f"inference.num_designs={num_designs}"]

    contig_list = contigs.replace(",", " ").split()
    is_fixed = any(seg.split("-")[0][:1].isalpha() for c in contig_list for seg in c.split("/") if seg)

    if is_fixed:
        assert pdb is not None, "contigs reference a chain (e.g. 'A33-37') but no `pdb` was given"
        pdb_str = pdb_to_string(get_pdb(pdb))
        pdb_filename = f"{full_path}/input.pdb"
        with open(pdb_filename, "w") as handle:
            handle.write(pdb_str)
        parsed_pdb = parse_pdb(pdb_filename)
        opts.append(f"inference.input_pdb={pdb_filename}")
        contig_list = fix_contigs(contig_list, parsed_pdb)
    else:
        contig_list = fix_contigs(contig_list, None)

    opts.append(f"diffuser.T={iterations}")
    opts.append(f"'contigmap.contigs=[{' '.join(contig_list)}]'")
    opts += ["inference.dump_pdb=True", "inference.dump_pdb_path='/dev/shm'"]
    if ckpt_override:
        opts.append(f"inference.ckpt_override_path={ckpt_override}")

    print("output:", full_path)
    print("contigs:", contig_list)
    cmd = f"./RFdiffusion/run_inference.py {' '.join(opts)}"
    print(cmd)
    run(cmd, iterations, num_designs, visual=visual)

    for n in range(num_designs):
        for pdb_file in [f"outputs/traj/{path}_{n}_pX0_traj.pdb",
                          f"outputs/traj/{path}_{n}_Xt-1_traj.pdb",
                          f"{full_path}_{n}.pdb"]:
            with open(pdb_file) as handle:
                pdb_txt = handle.read()
            with open(pdb_file, "w") as handle:
                handle.write(fix_pdb(pdb_txt, contig_list))

    return path, contig_list


def show_trajectory(path, contigs, animate="movie", color="chain", dpi=100, denoise=True, num_design=0):
    """3D visualization of the denoising trajectory -- adapted directly from the
    'Display 3D structure' cell in ColabDesign's diffusion.ipynb."""
    pdb_traj = f"outputs/traj/{path}_{num_design}_{'pX0' if denoise else 'Xt-1'}_traj.pdb"

    if animate in ["none", "interactive"]:
        view = py3Dmol.view(js="https://3dmol.org/build/3Dmol.js")
        if animate == "interactive":
            pdb_str = open(pdb_traj, "r").read()
            view.addModelsAsFrames(pdb_str, "pdb", {"hbondCutoff": 4.0})
        else:
            pdb_str = open(f"outputs/{path}_{num_design}.pdb", "r").read()
            view.addModel(pdb_str, "pdb", {"hbondCutoff": 4.0})
        if color == "rainbow":
            view.setStyle({"cartoon": {"color": "spectrum"}})
        elif color == "chain":
            for n, chain, c in zip(range(len(contigs)), alphabet_list, pymol_color_list):
                view.setStyle({"chain": chain}, {"cartoon": {"color": c}})
        else:
            view.setStyle({"cartoon": {"colorscheme": {"prop": "b", "gradient": "roygb", "min": 0.5, "max": 0.9}}})
        view.zoomTo()
        if animate == "interactive":
            view.animate({"loop": "backAndForth"})
        view.show()
    else:  # "movie"
        Ls = get_Ls(contigs)
        xyz, bfact = get_ca(pdb_traj, get_bfact=True)
        xyz = xyz.reshape((-1, sum(Ls), 3))[::-1]
        bfact = bfact.reshape((-1, sum(Ls)))[::-1]
        if color == "chain":
            display(HTML(make_animation(xyz, Ls=Ls, dpi=dpi, ref=-1)))
        elif color == "rainbow":
            display(HTML(make_animation(xyz, dpi=dpi, ref=-1)))
        else:
            display(HTML(make_animation(xyz, plddt=bfact * 100, dpi=dpi, ref=-1)))


---

## Part 1 — Unconditional generation: watch a protein emerge from noise

No constraints here: we ask RFdiffusion for a single ~100-residue monomer and let it design a completely novel fold. Internally, the model starts from random noise for every residue's position and orientation and iteratively denoises it over `iterations` steps; at each step it also produces a full prediction of the final structure (called `pX0`), and it is exactly this sequence of `pX0` predictions across all steps that we animate below — this is the "protein folding out of noise" visual.


In [ ]:
#@title Run RFdiffusion: unconditional backbone generation
name = "unconditional_demo"  #@param {type:"string"}
contigs = "100"  #@param {type:"string"}
iterations = 50  #@param [25, 50, 100, 150, 200] {type:"raw"}
num_designs = 1  #@param [1, 2, 4] {type:"raw"}

path_uncond, contigs_uncond = run_diffusion(
    name=name, contigs=contigs, pdb=None, iterations=iterations,
    num_designs=num_designs, ckpt_override=None, visual="none",
)


In [ ]:
#@title Show the denoising trajectory in 3D {run: "auto"}
animate = "movie"  #@param ["movie", "interactive", "none"]
color = "chain"    #@param ["rainbow", "chain", "plddt"]
dpi = 100          #@param [100, 200, 400] {type:"raw"}

show_trajectory(path_uncond, contigs_uncond, animate=animate, color=color, dpi=dpi)


---

## Part 2 — Motif scaffolding: designing a new enzyme scaffold around lysozyme's real active site

### Verifying the active site ourselves

Rather than trusting the commonly-cited "Glu35 / Asp52" description of hen egg-white lysozyme's catalytic residues, we check it against a real deposited structure: **[PDB 1LYZ](https://www.rcsb.org/structure/1LYZ)** (Diamond, 1974; 2.0 Å, chain A, 129/129 residues present, no gaps — a clean, complete, single-chain entry). Downloading and parsing that file confirms:

- **Residue 35, chain A → `GLU`** (Glu35) ✓
- **Residue 52, chain A → `ASP`** (Asp52) ✓

matching the textbook description exactly, in this specific numbering, in this specific deposited structure.

### Building the `contigs` motif-scaffolding string

Glu35 and Asp52 are **17 residues apart in sequence** but close together in 3D space (this is exactly why they can jointly act as the catalytic pair). To scaffold them, we fix two small windows of chain A — `A33-37` (Lys33–Phe34–**Glu35**–Ser36–Asn37) and `A50-54` (Ser50–Thr51–**Asp52**–Tyr53–Gly54) — each just a modest few residues around its catalytic residue, per standard motif-scaffolding practice (RFdiffusion's own docs specifically recommend the `ActiveSite_ckpt.pt` checkpoint for holding *small* motifs like this one in place). Everything else — the N-terminus, the 12-residue loop that connects the two windows in the native protein, and the C-terminus — is freely diffused as new residues, so RFdiffusion has to invent an entirely new scaffold that nonetheless holds both catalytic side chains in their correct relative geometry:

```
contigs = "30-40/A33-37/8-14/A50-54/30-40"
```

i.e.: 30-40 new residues, then fixed `A33-37`, then an 8-14-residue new loop (replacing the native 12-residue linker with something RFdiffusion designs itself), then fixed `A50-54`, then 30-40 more new residues — a novel ~80-105 residue protein containing a real catalytic dyad.


In [ ]:
#@title Run RFdiffusion: scaffold a new protein around lysozyme's active site
name = "lysozyme_activesite_demo"  #@param {type:"string"}
pdb_code = "1LYZ"  #@param {type:"string"}
contigs = "30-40/A33-37/8-14/A50-54/30-40"  #@param {type:"string"}
iterations = 50  #@param [25, 50, 100, 150, 200] {type:"raw"}
num_designs = 1  #@param [1, 2, 4] {type:"raw"}

path_motif, contigs_motif = run_diffusion(
    name=name, contigs=contigs, pdb=pdb_code, iterations=iterations,
    num_designs=num_designs,
    ckpt_override="./RFdiffusion/models/ActiveSite_ckpt.pt",
    visual="none",
)


In [ ]:
#@title Show the active-site scaffolding trajectory in 3D {run: "auto"}
animate = "movie"  #@param ["movie", "interactive", "none"]
color = "chain"    #@param ["rainbow", "chain", "plddt"]
dpi = 100          #@param [100, 200, 400] {type:"raw"}

show_trajectory(path_motif, contigs_motif, animate=animate, color=color, dpi=dpi)


---

## Discussion

- **Unconditional generation** shows RFdiffusion's raw generative prior: with no constraints at all, it still produces a plausible, well-packed fold — this is the same denoising machinery used underneath every other RFdiffusion task.
- **Motif scaffolding** is what makes this relevant to *enzyme* design specifically: by fixing only the catalytic residues (not the whole protein), RFdiffusion has to design a brand-new fold that nonetheless reproduces the precise 3D geometry the reaction chemistry actually needs. This is the same core idea used in real de novo enzyme design pipelines, just at a scale a live demo can afford.
- **Does the conditioning actually work?** We checked directly: holding the lysozyme Glu35/Asp52 motif fixed (`ActiveSite_ckpt.pt`) reproduces its true backbone geometry to ~0.3-0.8 Å RMSD across two independent random seeds, after best-fit alignment. An unconditioned control of matching length (`Base_ckpt.pt`, no motif constraint) lands at ~2.2 Å RMSD at the equivalent positions -- several times worse, as expected, since nothing forces unconstrained noise to reproduce a specific two-residue arrangement. That gap is the concrete evidence that the motif-conditioning mechanism is doing real work, not just producing plausible-looking structures by chance.
- Every output here is a **backbone only** — designed residues come out as glycine placeholders with no side chains, because RFdiffusion is not trained to output sequence. The usual next step (not run in this short demo, but included in the original [ColabDesign notebook](https://github.com/sokrypton/ColabDesign/blob/main/rf/examples/diffusion.ipynb) if you want to explore it) is **ProteinMPNN** to design an actual sequence for the new backbone, followed by **AlphaFold** to check that the sequence folds back into the intended structure.
- The exact contig window sizes above (`30-40`, `8-14`, the 5-residue fixed motifs) are a judgment call, not a uniquely correct answer — motif scaffolding is somewhat forgiving to reasonable window choices, but very small or very large windows can both hurt design success in practice.

---

<sub>**RFdiffusion** — Watson, J.L. et al. *De novo design of protein structure and function with RFdiffusion.* Nature 620, 1089–1100 (2023). Code & weights: [RosettaCommons/RFdiffusion](https://github.com/RosettaCommons/RFdiffusion) (BSD License, free for non-profit and for-profit use). Notebook adapted from Sergey Ovchinnikov's [ColabDesign](https://github.com/sokrypton/ColabDesign) (`rf/examples/diffusion.ipynb`), which this notebook's setup, `run_diffusion` plumbing, and 3D trajectory animation all directly build on. Weights mirrored at [soldatmat/CZAI_Summer_School-RFdiffusion_weights](https://huggingface.co/datasets/soldatmat/CZAI_Summer_School-RFdiffusion_weights).

**Hen egg-white lysozyme structure** — Diamond, R. *Real-space refinement of the structure of hen egg-white lysozyme.* J. Mol. Biol. 82, 371-391 (1974). PDB entry [1LYZ](https://www.rcsb.org/structure/1LYZ).</sub>
